In [1]:
#All the imports
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.stats import norm
from scipy.stats import kurtosis
from scipy.ndimage import maximum_filter, label
import multiprocessing as mp
import os

In [2]:
#Function Definitions
def partial_x_2d(f,dx):
    """Compute ∂f/∂x using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dx = np.zeros_like(f)
    
    # Central difference in x-direction (axis=0)
    df_dx[1:Nx-1, :] = (f[2:Nx, :] - f[0:Nx-2, :]) / (2 * dx)
    
    # Periodic boundaries (x-direction)
    df_dx[0, :] = (f[1, :] - f[-1, :]) / (2 * dx)
    df_dx[-1, :] = (f[0, :] - f[-2, :]) / (2 * dx)
    
    return df_dx

def partial_y_2d(f,dy):
    """Compute ∂f/∂y using 2nd-order central differences with periodic BCs."""
    Nx, Ny = f.shape
    df_dy = np.zeros_like(f)
    
    # Central difference in y-direction (axis=1)
    df_dy[:, 1:Ny-1] = (f[:, 2:Ny] - f[:, 0:Ny-2]) / (2 * dy)
    
    # Periodic boundaries (y-direction)
    df_dy[:, 0] = (f[:, 1] - f[:, -1]) / (2 * dy)
    df_dy[:, -1] = (f[:, 0] - f[:, -2]) / (2 * dy)
    
    return df_dy

def partial_x_fft(f, dx):
    """
    Calculate ∂f/∂x using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dx (float): Grid spacing in the x-direction.

    Returns:
        df_dx (2D np.array): Partial derivative ∂f/∂x.
    """
    nx, ny = f.shape
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi  # Wavenumbers in x
    kx = kx[:, np.newaxis]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dx_hat = 1j * kx * f_hat
    df_dx = np.fft.ifft2(df_dx_hat).real

    return np.array(df_dx)

def partial_y_fft(f, dy):
    """
    Calculate ∂f/∂y using FFT on a 2D grid.

    Parameters:
        f (2D np.array): Function values on a 2D grid (shape: Nx × Ny).
        dy (float): Grid spacing in the y-direction.

    Returns:
        df_dy (2D np.array): Partial derivative ∂f/∂y.
    """
    nx, ny = f.shape
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi  # Wavenumbers in y
    ky = ky[np.newaxis, :]                     # Make it 2D for broadcasting

    f_hat = np.fft.fft2(f)
    df_dy_hat = 1j * ky * f_hat
    df_dy = np.fft.ifft2(df_dy_hat).real

    return df_dy

def index_of_just_smaller(arr, value):
    arr = np.asarray(arr)
    mask = arr < value
    if not np.any(mask):
        return None  # No smaller value exists
    return np.argmax(np.where(mask, arr, -np.inf))

In [3]:
# List of simulation directories
base_root = "/home/UNN/w24021992/NAS/simulations_beta_scan"

sim_dirs = [

    # f"{base_root}/beta_p_0.0625/beta_e_0.25",
    # f"{base_root}/beta_p_0.0625/beta_e_1",
    # f"{base_root}/beta_p_0.0625/beta_e_4",

    # f"{base_root}/beta_p_0.25/beta_e_1",
    # f"{base_root}/beta_p_0.25/beta_e_4",
    # f"{base_root}/beta_p_0.25/beta_e_16",

    # f"{base_root}/beta_p_1/beta_e_0.0625",
    # f"{base_root}/beta_p_1/beta_e_0.25",
    # f"{base_root}/beta_p_1/beta_e_1",
    # f"{base_root}/beta_p_1/beta_e_4",
    # f"{base_root}/beta_p_1/beta_e_16",

    # f"{base_root}/beta_p_4/beta_e_0.0625",
    # f"{base_root}/beta_p_4/beta_e_0.25",
    # f"{base_root}/beta_p_4/beta_e_1",
    # # f"{base_root}/beta_p_4/beta_e_4",
    # f"{base_root}/beta_p_4/beta_e_16",
    
    # f"{base_root}/beta_p_16/beta_e_0.0625",
    # f"{base_root}/beta_p_16/beta_e_0.25",
    # f"{base_root}/beta_p_16/beta_e_16",

    f"{base_root}/beta_p_1/beta_e_1"
]

dx = 0.0625
dy = 0.0625

In [4]:
def process_sim_dir(sim_dir):
    J_rms = []
    print(f"{sim_dir}", flush=True)
    for i in range(0, 300):
        print(i)
        File_Bx = f'{sim_dir}/Bx_ApJ_t{i}.h5'
        File_By = f'{sim_dir}/By_ApJ_t{i}.h5'

        with h5py.File(File_Bx, 'r') as fBx, h5py.File(File_By, 'r') as fBy:
            data_Bx = fBx['DS1'][:].T
            data_By = fBy['DS1'][:].T

        J_par_FT = partial_x_fft(data_By, dx) - partial_y_fft(data_Bx, dy)
        rms = np.sqrt(np.mean(J_par_FT**2))
        J_rms.append(rms)

    return sim_dir, J_rms


if __name__ == "__main__":
    with mp.Pool(processes=32) as pool:   # adjust to number of CPU cores
        results = pool.map(process_sim_dir, sim_dirs)

    all_J_rms = {sim_dir: J_rms for sim_dir, J_rms in results}

/home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_1
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
26

In [7]:
timesteps = np.arange(0, 300)

for sim_dir, J_rms in all_J_rms.items():

    J_rms = np.asarray(J_rms)

    # --- Find global peak ---
    imax = np.argmax(J_rms)
    t_peak = timesteps[imax]
    J_peak = J_rms[imax]

    # --- Create plots directory ---
    plots_dir = os.path.join(sim_dir, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # --- Plot ---
    plt.figure(figsize=(10, 6))
    plt.plot(timesteps, J_rms, lw=2, label=r'$J_{\mathrm{rms}}$')

    # Vertical line at peak
    plt.axvline(t_peak, linestyle='--', linewidth=1.5)

    # Peak marker
    plt.scatter(t_peak, J_peak, zorder=5)

    # Annotation
    plt.text(
        t_peak,
        J_peak,
        f'  Peak\n  t = {t_peak}\n  $J_{{rms}}$ = {J_peak:.3e}',
        verticalalignment='bottom',
        horizontalalignment='left',
        bbox=dict(boxstyle="round", alpha=0.25)
    )

    plt.xlabel('Timestep')
    plt.ylabel(r'$J_{\mathrm{rms}}$')
    plt.title(rf'$J_{{\mathrm{{rms}}}}$ vs Time')
    # plt.grid(True)
    plt.tight_layout()

    # --- Save ---
    save_path = os.path.join(plots_dir, "Jrms.png")
    plt.savefig(save_path, dpi=300)
    plt.close()

    print(f"Saved Jrms plot with peak annotation: {save_path}")

Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.0625/beta_e_0.25/plots/Jrms.png
Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.0625/beta_e_1/plots/Jrms.png
Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.0625/beta_e_4/plots/Jrms.png
Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.25/beta_e_1/plots/Jrms.png
Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.25/beta_e_4/plots/Jrms.png
Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_0.25/beta_e_16/plots/Jrms.png
Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_0.0625/plots/Jrms.png
Saved Jrms plot with peak annotation: /home/UNN/w24021992/NAS/simulations_beta_scan/beta_p_1/beta_e_0.25/plots/Jrms.png
Saved Jrms plot with peak an